# KITP Effect of Future Arctic Sea Ice Loss on the Greenland Ice Sheet 

In [ ]:
from functools import partial
from pathlib import Path

import dask
import xarray as xr
import pint_xarray
import matplotlib.pylab as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from dask.diagnostics import ProgressBar
from pism_terra.processing import preprocess_config_rgi

import warnings                                                                                                                                                    
warnings.filterwarnings("ignore", message="Increasing number of chunks")     
from cmap import Colormap
cm = Colormap('tol:bright').to_matplotlib() 

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
delta_coder = xr.coders.CFTimedeltaCoder()

## Ice-sheet wide scalars

In [ ]:
%%time
scalar_lowres_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_1500m/output/spatial/").glob("fldsum_*.nc"))
scalar_hires_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_900m/output/spatial/").glob("fldsum_*.nc"))

def load_dataset(filename_or_obj: str | Path) -> xr.Dataset:
    with ProgressBar():
        ds = xr.open_mfdataset(filename_or_obj,
                                      preprocess=partial(preprocess_config_rgi, 
                                                         exp_regexp=r"id_(.+?)_uq_",                                                                                                                     
                                                         rgi_regexp="uq_(.+?)_",   
                                                         rgi_dim="uq_id",
                                                         exp_dim="exp_id"), 
                                      engine="netcdf4",
                                      parallel=True,
                                      chunks=None,
                                      data_vars="minimal",
                                      join="outer",
                                      decode_times=time_coder,
                                      decode_timedelta=delta_coder).pint.quantify()
    ds["exp_id"] = ds["exp_id"].str.replace("CESM1-WACCM-SC_", "") 
 
    return ds

with dask.config.set(**{"array.slicing.split_large_chunks": False}):
    scalar_lowres = load_dataset(scalar_lowres_files)
    # scalar_hires = load_dataset(scalar_hires_files)

In [ ]:
pctls = [0.05, 0.95]
rolling_window = 1
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}


In [ ]:
basin = "GIS"
years = np.arange(300) + 2025
colors = cm(np.linspace(0, 1, scalar_lowres.exp_id.size))
with mpl.rc_context(rc=rc_params):

    fig, axs = plt.subplots(3, 1, sharex=True, figsize=(6.4, 6.4))
    for k, (exp_id, _ds) in enumerate(scalar_lowres.groupby("exp_id")):
        c = colors[k]
        _ds = _ds.sel({"basin": basin}).squeeze()                                                                                                                                                                       
        _ds = _ds.assign_coords(time=("time", years))
        _ice_mass = _ds.ice_mass.pint.to("Gt").pint.dequantify()
        _ice_mass -= _ice_mass.isel({"time": 0})
        _ice_mass_pctls = {}
        _surface_flux = _ds.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt/yr").rolling(time=rolling_window).mean().pint.dequantify()
        _surface_flux_pctls = {}
        _grounding_line_flux = (_ds.tendency_of_ice_mass - _ds.tendency_of_ice_mass_due_to_surface_mass_flux).pint.to("Gt/yr").rolling(time=rolling_window).mean().pint.dequantify()
        _grounding_line_flux_pctls = {}
        for pctl in pctls:
            _ice_mass_pctls[pctl] = _ice_mass.quantile(pctl, dim="uq_id", skipna=True)
            _surface_flux_pctls[pctl] = _surface_flux.quantile(pctl, dim="uq_id", skipna=True)   
            _grounding_line_flux_pctls[pctl] = _grounding_line_flux.quantile(pctl, dim="uq_id", skipna=True)   
        axs[0].fill_between(_ice_mass.time, _ice_mass_pctls[pctls[0]], _ice_mass_pctls[pctls[1]], color=c, alpha=0.5, lw=0, label=exp_id)
        axs[1].fill_between(_surface_flux.time, _surface_flux_pctls[pctls[0]], _surface_flux_pctls[pctls[1]], color=c, alpha=0.5, lw=0)
        axs[2].fill_between(_grounding_line_flux.time, _grounding_line_flux_pctls[pctls[0]], _grounding_line_flux_pctls[pctls[1]], color=c, alpha=0.5, lw=0)
        # _ice_mass.plot(hue="uq_id", color=c, ax=axs[0], lw=0.5, add_legend=False)
        # _surface_flux.plot(hue="uq_id", color=c, ax=axs[1], lw=0.5, add_legend=False)
        # _grounding_line_flux.plot(hue="uq_id", color=c, ax=axs[2], lw=0.5, add_legend=False)
    axs[0].set_ylabel("Cumulative mass change (Gt)")
    axs[1].set_ylabel("Surface mass flux (Gt/yr)")
    axs[2].set_ylabel("Grounding line flux (Gt/yr)")
    for ax in axs.flat:
        ax.set_xlim(years[0], years[-1])
    # Create a legend outside the figure at the bottom middle
    handles, labels = axs[0].get_legend_handles_labels()
    legend_main = fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.3, 0.95), ncol=1)
    legend_main.set_title("5-95th pctl")
    legend_main.get_frame().set_linewidth(0.0)
    legend_main.get_frame().set_alpha(0.0)
    fig.tight_layout()
    plt.show()
    fig.savefig("pism_kitp.png", dpi=300)
    fig.savefig("pism_kitp.pdf")
    plt.close()
    del fig

In [ ]:
colors = cm(np.linspace(0, 1, scalar_lowres.exp_id.size))
with mpl.rc_context(rc=rc_params):

    fig, axs = plt.subplots(3, 3, sharex=True, sharey=False, figsize=(6.4, 3.2))
    
    for basin_name, ax in zip(['GIS', 'CE', 'CW', 'NE', 'NO', 'NW', 'SE', 'SW'], axs.flat):
        basin = scalar_lowres.sel(basin=basin_name)
        for k, (exp_id, _ds) in enumerate(basin.groupby("exp_id")):
            legend = bool(k == 0)
            c = colors[k]
            _ds = _ds.squeeze()                                                                                                                                                                       
            _ds = _ds.assign_coords(time=("time", np.arange(300) + 2025))
            _ice_mass = _ds.ice_mass.pint.to("Gt").pint.dequantify()
            _ice_mass -= _ice_mass.isel({"time": 0})
            _ice_mass_pctls = {}
            for pctl in pctls:
                _ice_mass_pctls[pctl] = _ice_mass.quantile(pctl, dim="uq_id", skipna=True)
            ax.fill_between(_ice_mass.time, _ice_mass_pctls[pctls[0]], _ice_mass_pctls[pctls[1]], color=c, alpha=0.5, lw=0, label=exp_id)
            ax.set_title(basin_name)
    axs[-1, -1].set_visible(False)  
    handles, labels = axs[0,0].get_legend_handles_labels()
    legend_main = fig.legend(handles, labels, loc="lower right", bbox_to_anchor=(0.98, 0.10), ncol=1)
    legend_main.set_title("5-95th pctl")
    legend_main.get_frame().set_linewidth(0.0)
    legend_main.get_frame().set_alpha(0.0)
    fig.supylabel("Cumulative mass change since 2025 (Gt)")                                                                                                                                         
    fig.supxlabel("Years") 
    fig.tight_layout()
    plt.show()
    fig.savefig("kitp_by_basin.pdf")
    plt.close()
    del fig